# 🫀 Cardiomegaly Prediction — DenseNet-121 vs EfficientNet-B3
**Paper**: Cardiomegaly Prediction Using Deep Learning — Aaditya Sharma et al., MIET

**Dataset**: [Cardiomegaly Disease Prediction Using CNN](https://www.kaggle.com/datasets/rahimanshu/cardiomegaly-disease-prediction-using-cnn)

### Before you start:
1. `Runtime → Change runtime type → T4 GPU`
2. Run **Cell 1** (install) → **Runtime → Restart session**
3. Run **Cell 2 onwards**

## Cell 1 — Install (then restart runtime!)

In [ ]:
!pip install -q kaggle timm grad-cam
print("✅ Done! Now: Runtime → Restart session → then run from Cell 2")

## Cell 2 — Imports (run after restarting)

In [ ]:
import os, random, warnings, zipfile, shutil, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

import torch, torch.nn as nn, torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
import timm

# Mixed precision — compatible with all PyTorch versions
try:
    from torch.amp import GradScaler, autocast
    def amp_autocast(): return autocast(device_type='cuda')
except ImportError:
    from torch.cuda.amp import GradScaler, autocast
    def amp_autocast(): return autocast()

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    roc_curve, classification_report
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {DEVICE}')
if torch.cuda.is_available(): print(f'   GPU: {torch.cuda.get_device_name(0)}')
print(f'   NumPy {np.__version__} | PyTorch {torch.__version__}')

## Cell 3 — Kaggle Credentials

In [ ]:
import os, json

# ── Paste your credentials here ──────────────────────────────────────────────
KAGGLE_USERNAME = 'your_kaggle_username'   # ← change this
KAGGLE_KEY      = 'your_kaggle_api_token'  # ← paste your KGAT_xxx... token here
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ Kaggle credentials saved!')

## Cell 4 — Download Dataset

In [ ]:
import os, zipfile

DATA_ROOT = '/content/cardiomegaly_data'
os.makedirs(DATA_ROOT, exist_ok=True)

print('⬇️  Downloading dataset...')
!kaggle datasets download -d rahimanshu/cardiomegaly-disease-prediction-using-cnn \
    --path {DATA_ROOT} -q

# Extract
zip_files = [f for f in os.listdir(DATA_ROOT) if f.endswith('.zip')]
print(f'📦 Extracting {zip_files}...')
for zf in zip_files:
    with zipfile.ZipFile(os.path.join(DATA_ROOT, zf)) as z:
        z.extractall(DATA_ROOT)

# Show what we got
print('\n📁 Dataset structure:')
for root, dirs, files in os.walk(DATA_ROOT):
    depth = root.replace(DATA_ROOT, '').count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth < 3:
        for f in files[:3]:
            print(f'{indent}  {f}')
        if len(files) > 3:
            print(f'{indent}  ... ({len(files)} files total)')

## Cell 5 — Build DataFrame from Folder Structure

In [ ]:
import os, glob
import pandas as pd

DATA_ROOT  = '/content/cardiomegaly_data'
IMAGE_EXTS = ('*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG')

# folder name → label (covers all known naming conventions)
LABEL_MAP = {
    'true': 1,  'false': 0,
    'cardiomegaly': 1, 'normal': 0,
    'no_finding': 0,   'nofinding': 0,
    'positive': 1,     'negative': 0,
    'abnormal': 1,     'healthy': 0,
    '1': 1, '0': 0
}

def collect_images(root):
    rows = []
    for dirpath, _, _ in os.walk(root):
        folder_name = os.path.basename(dirpath).lower().strip()
        if folder_name not in LABEL_MAP:
            continue
        label = LABEL_MAP[folder_name]
        files = []
        for ext in IMAGE_EXTS:
            files.extend(glob.glob(os.path.join(dirpath, ext)))
        for fpath in files:
            rows.append({'filepath': fpath, 'label': label})
        if files:
            print(f'  {os.path.relpath(dirpath, DATA_ROOT)}  →  label={label}  ({len(files)} images)')
    return pd.DataFrame(rows)

print('Scanning dataset...')
df = collect_images(DATA_ROOT)

if len(df) == 0:
    print('\n⚠️  No images matched. All folders with images:')
    for dirpath, dirs, files in os.walk(DATA_ROOT):
        imgs = [f for f in files if f.lower().endswith(('.png','.jpg','.jpeg'))]
        if imgs:
            print(f'  {dirpath}  ({len(imgs)} images)')
    raise ValueError('Update LABEL_MAP above with the folder names shown')

print(f'\nTotal images found: {len(df)}')
vc = df['label'].value_counts().sort_index()
print(f'  Cardiomegaly (1): {vc.get(1, 0)}')
print(f'  Normal       (0): {vc.get(0, 0)}')

# Balance
n = min(vc.get(0, 0), vc.get(1, 0))
df = pd.concat([
    df[df['label']==1].sample(n, random_state=SEED),
    df[df['label']==0].sample(n, random_state=SEED)
]).reset_index(drop=True)
print(f'\n✅ Balanced: {len(df)} images ({n} per class)')


## Cell 6 — Train/Val/Test Split & Sample Visualization

In [ ]:
# Split: 70% train, 15% val, 15% test
df_train, df_temp = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
df_val,   df_test = train_test_split(df_temp, test_size=0.50, stratify=df_temp['label'], random_state=SEED)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)
print(f'Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')

# Class distribution plot
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Dataset Overview', fontsize=13, fontweight='bold')

counts = df['label'].value_counts().sort_index()
axes[0].bar(['Normal', 'Cardiomegaly'], counts.values,
            color=['#2196F3', '#F44336'], edgecolor='black', width=0.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')
axes[0].set_title('Class Distribution'); axes[0].set_ylabel('Count')

split_labels = ['Train', 'Val', 'Test']
split_sizes  = [len(df_train), len(df_val), len(df_test)]
axes[1].pie(split_sizes, labels=split_labels, autopct='%1.0f%%',
            colors=['#4CAF50','#FF9800','#9C27B0'], startangle=90)
axes[1].set_title('Train / Val / Test Split')
plt.tight_layout()
plt.savefig('/content/dataset_overview.png', dpi=130)
plt.show()

# Show sample images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Chest X-rays', fontsize=13, fontweight='bold')
for row_idx, (label_val, label_name, color) in enumerate([
    (0, 'Normal',        '#2196F3'),
    (1, 'Cardiomegaly',  '#F44336')
]):
    samples = df[df['label'] == label_val].sample(5, random_state=SEED)
    for col_idx, (_, row) in enumerate(samples.iterrows()):
        ax = axes[row_idx, col_idx]
        try:
            img = Image.open(row['filepath']).convert('L')
            ax.imshow(img, cmap='gray')
        except:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(label_name, color=color, fontweight='bold', fontsize=9)
        ax.axis('off')
plt.tight_layout()
plt.savefig('/content/sample_images.png', dpi=130)
plt.show()
print('Plots saved!')

## Cell 7 — Dataset Class & DataLoaders

In [ ]:
IMG_SIZE   = 224
BATCH_SIZE = 32
MEAN = [0.485, 0.456, 0.406]   # ImageNet stats
STD  = [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.Grayscale(num_output_channels=3),
    T.RandomHorizontalFlip(0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD)
])

eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(MEAN, STD)
])

class XRayDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row['filepath']).convert('RGB')
        except:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), 0)
        if self.transform: img = self.transform(img)
        return img, torch.tensor(int(row['label']), dtype=torch.long)

train_loader = DataLoader(XRayDataset(df_train, train_tf), batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(XRayDataset(df_val,   eval_tf),  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(XRayDataset(df_test,  eval_tf),  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_loader)} batches | Val: {len(val_loader)} | Test: {len(test_loader)}')
print('✅ DataLoaders ready!')

## Cell 8 — Build Models

In [ ]:
NUM_CLASSES = 2

def make_head(in_features):
    return nn.Sequential(
        nn.BatchNorm1d(in_features),
        nn.Dropout(0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, NUM_CLASSES)
    )

def build_efficientnet(freeze=True):
    m = timm.create_model('efficientnet_b3', pretrained=True, num_classes=0)
    if freeze:
        for p in m.parameters(): p.requires_grad = False
    m.classifier = make_head(m.num_features)
    return m

def build_densenet(freeze=True):
    m = models.densenet121(weights='IMAGENET1K_V1')
    if freeze:
        for p in m.features.parameters(): p.requires_grad = False
    m.classifier = make_head(m.classifier.in_features)
    return m

def count_params(m):
    total     = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

for name, fn in [('EfficientNet-B3', build_efficientnet), ('DenseNet-121', build_densenet)]:
    t, tr = count_params(fn().to(DEVICE))
    print(f'{name:20s} → Total: {t/1e6:.1f}M params | Trainable: {tr/1e6:.1f}M')
print('✅ Model builders ready!')

## Cell 9 — Training Engine

In [ ]:
def run_epoch(model, loader, optimizer, criterion, scaler, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if training: optimizer.zero_grad()

            with amp_autocast():
                out  = model(imgs)
                loss = criterion(out, labels)

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * imgs.size(0)
            probs  = torch.softmax(out, dim=1)[:, 1]
            preds  = out.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return (total_loss / total, correct / total,
            np.array(all_preds), np.array(all_labels), np.array(all_probs))


def train_model(build_fn, model_name, n_epochs=25, lr=1e-3, unfreeze_at=10):
    model = build_fn(freeze=True).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                             lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-6)
    scaler    = GradScaler()

    history = {'tl': [], 'vl': [], 'ta': [], 'va': [], 'lr': []}
    best_acc, best_wts = 0.0, None

    print(f'\n{"="*58}')
    print(f'  Training {model_name}')
    print(f'{"="*58}')
    print(f'  {"Epoch":>6}  {"TrainLoss":>10}  {"TrainAcc":>10}  {"ValLoss":>8}  {"ValAcc":>8}')
    print(f'  {"-"*54}')

    for epoch in range(1, n_epochs + 1):

        # Unfreeze backbone at unfreeze_at for fine-tuning
        if epoch == unfreeze_at:
            print(f'  🔓 Epoch {epoch}: Unfreezing all layers for fine-tuning')
            for p in model.parameters(): p.requires_grad = True
            optimizer = optim.AdamW(model.parameters(), lr=lr * 0.05, weight_decay=1e-4)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=n_epochs - unfreeze_at + 1, eta_min=1e-7)

        tl, ta, _, _, _          = run_epoch(model, train_loader, optimizer, criterion, scaler, True)
        vl, va, vp, vlab, vprob  = run_epoch(model, val_loader,   optimizer, criterion, scaler, False)
        scheduler.step()

        history['tl'].append(tl); history['vl'].append(vl)
        history['ta'].append(ta); history['va'].append(va)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        star = ''
        if va > best_acc:
            best_acc = va
            best_wts = {k: v.clone() for k, v in model.state_dict().items()}
            star = ' ⭐'

        print(f'  {epoch:>6}  {tl:>10.4f}  {ta*100:>9.2f}%  {vl:>8.4f}  {va*100:>7.2f}%{star}')

    model.load_state_dict(best_wts)
    print(f'\n  ✅ Best Val Acc: {best_acc*100:.2f}%')
    return model, history

print('✅ Training engine ready!')

## Cell 10 — Train EfficientNet-B3

In [ ]:
eff_model, eff_hist = train_model(
    build_efficientnet, 'EfficientNet-B3',
    n_epochs=25, lr=1e-3, unfreeze_at=10
)
torch.save(eff_model.state_dict(), '/content/efficientnet_b3_best.pth')
print('Model saved!')

## Cell 11 — Train DenseNet-121

In [ ]:
dens_model, dens_hist = train_model(
    build_densenet, 'DenseNet-121',
    n_epochs=25, lr=1e-3, unfreeze_at=10
)
torch.save(dens_model.state_dict(), '/content/densenet121_best.pth')
print('Model saved!')

## Cell 12 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Training History: EfficientNet-B3 vs DenseNet-121', fontsize=13, fontweight='bold')
epochs = range(1, len(eff_hist['tl']) + 1)

# Loss
axes[0].plot(epochs, eff_hist['tl'],  'b-',  lw=2, label='EfficientNet Train')
axes[0].plot(epochs, eff_hist['vl'],  'b--', lw=2, label='EfficientNet Val')
axes[0].plot(epochs, dens_hist['tl'], 'r-',  lw=2, label='DenseNet Train')
axes[0].plot(epochs, dens_hist['vl'], 'r--', lw=2, label='DenseNet Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs, [v*100 for v in eff_hist['ta']],  'b-',  lw=2, label='EfficientNet Train')
axes[1].plot(epochs, [v*100 for v in eff_hist['va']],  'b--', lw=2, label='EfficientNet Val')
axes[1].plot(epochs, [v*100 for v in dens_hist['ta']], 'r-',  lw=2, label='DenseNet Train')
axes[1].plot(epochs, [v*100 for v in dens_hist['va']], 'r--', lw=2, label='DenseNet Val')
axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('Epoch'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

# LR schedule
axes[2].plot(epochs, eff_hist['lr'],  'b-', lw=2, label='EfficientNet')
axes[2].plot(epochs, dens_hist['lr'], 'r-', lw=2, label='DenseNet')
axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch')
axes[2].set_yscale('log'); axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show(); print('Saved!')

## Cell 13 — Evaluate on Test Set

In [ ]:
CLASS_NAMES = ['Normal', 'Cardiomegaly']
criterion   = nn.CrossEntropyLoss()
scaler_dummy = GradScaler()

def evaluate_model(model, model_name):
    _, acc, preds, labels, probs = run_epoch(
        model, test_loader, None, criterion, scaler_dummy, training=False)
    metrics = {
        'Accuracy':  accuracy_score(labels, preds),
        'Precision': precision_score(labels, preds, zero_division=0),
        'Recall':    recall_score(labels, preds, zero_division=0),
        'F1':        f1_score(labels, preds, zero_division=0),
        'AUC':       roc_auc_score(labels, probs)
    }
    print(f'\n── {model_name} ──────────────────────────')
    for k, v in metrics.items():
        print(f'  {k:<12}: {v:.4f}  ({v*100:.2f}%)')
    print(f'\n{classification_report(labels, preds, target_names=CLASS_NAMES)}')
    return metrics, preds, labels, probs

eff_metrics,  eff_preds,  eff_labels,  eff_probs  = evaluate_model(eff_model,  'EfficientNet-B3')
dens_metrics, dens_preds, dens_labels, dens_probs = evaluate_model(dens_model, 'DenseNet-121')

## Cell 14 — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Confusion Matrices on Test Set', fontsize=13, fontweight='bold')

for ax, preds, labels, title, cmap in [
    (axes[0], eff_preds,  eff_labels,  'EfficientNet-B3', 'Blues'),
    (axes[1], dens_preds, dens_labels, 'DenseNet-121',    'Reds')
]:
    cm = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=1, linecolor='white', cbar=False,
                annot_kws={'size': 14, 'weight': 'bold'})
    ax.set_title(f'{title}\nTP={tp}  TN={tn}  FP={fp}  FN={fn}',
                 fontweight='bold', fontsize=10)
    ax.set_xlabel('Predicted', fontweight='bold')
    ax.set_ylabel('Actual', fontweight='bold')

plt.tight_layout()
plt.savefig('/content/confusion_matrices.png', dpi=150)
plt.show(); print('Saved!')

## Cell 15 — ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for probs, labels, name, color in [
    (eff_probs,  eff_labels,  'EfficientNet-B3', '#1565C0'),
    (dens_probs, dens_labels, 'DenseNet-121',    '#C62828')
]:
    fpr, tpr, _ = roc_curve(labels, probs)
    auc = roc_auc_score(labels, probs)
    ax.plot(fpr, tpr, color=color, lw=2.5, label=f'{name}  (AUC = {auc:.3f})')

ax.plot([0,1],[0,1],'k--', lw=1.5, alpha=0.5)
ax.fill_between([0,1],[0,1], alpha=0.03, color='gray')
ax.set_xlabel('False Positive Rate', fontweight='bold')
ax.set_ylabel('True Positive Rate', fontweight='bold')
ax.set_title('ROC Curves — Cardiomegaly Detection', fontweight='bold', fontsize=13)
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/roc_curves.png', dpi=150)
plt.show(); print('Saved!')

## Cell 16 — Metrics Comparison Chart

In [ ]:
metric_keys = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
eff_vals  = [eff_metrics[k]  for k in metric_keys]
dens_vals = [dens_metrics[k] for k in metric_keys]

x = np.arange(len(metric_keys))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, eff_vals,  w, label='EfficientNet-B3', color='#1565C0', edgecolor='black')
b2 = ax.bar(x + w/2, dens_vals, w, label='DenseNet-121',    color='#C62828', edgecolor='black')
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.008,
            f'{b.get_height()*100:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(metric_keys, fontsize=11)
ax.set_ylim(0, 1.12); ax.set_ylabel('Score', fontsize=12)
ax.set_title('EfficientNet-B3 vs DenseNet-121 — Test Set Metrics',
             fontweight='bold', fontsize=13)
ax.legend(fontsize=11); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/metrics_comparison.png', dpi=150)
plt.show(); print('Saved!')

## Cell 17 — Grad-CAM Visualization

In [ ]:
import cv2

def gradcam(model, img_tensor, target_layer):
    grads, acts = [], []
    def fwd_hook(m, i, o): acts.append(o); o.register_hook(lambda g: grads.append(g))
    h = target_layer.register_forward_hook(fwd_hook)
    model.eval()
    out = model(img_tensor)
    model.zero_grad()
    out[0, out.argmax(1).item()].backward()
    h.remove()
    w = grads[0].cpu().detach().mean(dim=[2,3], keepdim=True)
    cam = torch.relu((w * acts[0].cpu().detach()).sum(1)).squeeze().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cv2.resize(cam, (IMG_SIZE, IMG_SIZE)), out.argmax(1).item()

def show_gradcam(model, model_name, target_layer, n=6):
    samples = df_test.sample(n, random_state=SEED)
    fig, axes = plt.subplots(2, n, figsize=(n*3, 6))
    fig.suptitle(f'Grad-CAM: {model_name}', fontsize=12, fontweight='bold')
    for i, (_, row) in enumerate(samples.iterrows()):
        img_pil = Image.open(row['filepath']).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        tensor  = eval_tf(img_pil).unsqueeze(0).to(DEVICE)
        cam, pred = gradcam(model, tensor, target_layer)
        true_lbl  = CLASS_NAMES[int(row['label'])]
        pred_lbl  = CLASS_NAMES[pred]
        color     = 'green' if pred == int(row['label']) else 'red'
        # Row 0: original
        axes[0,i].imshow(np.array(img_pil.convert('L')), cmap='gray')
        axes[0,i].set_title(f'True: {true_lbl}', fontsize=8); axes[0,i].axis('off')
        # Row 1: Grad-CAM overlay
        hm = cv2.applyColorMap(np.uint8(255*cam), cv2.COLORMAP_JET)
        hm = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB)
        base = np.stack([np.array(img_pil.convert('L'))]*3, axis=2).astype(np.float32)
        overlay = np.clip(0.5*base + 0.5*hm.astype(np.float32), 0, 255).astype(np.uint8)
        axes[1,i].imshow(overlay)
        axes[1,i].set_title(f'Pred: {pred_lbl}', fontsize=8, color=color, fontweight='bold')
        axes[1,i].axis('off')
    plt.tight_layout()
    fname = f'/content/gradcam_{model_name.lower().replace("-","_")}.png'
    plt.savefig(fname, dpi=130); plt.show(); print(f'Saved: {fname}')

# Target layers for Grad-CAM
eff_layer  = list(eff_model.blocks.children())[-1][-1].conv_pwl
dens_layer = dens_model.features.denseblock4.denselayer16.conv2

show_gradcam(eff_model,  'EfficientNet-B3', eff_layer)
show_gradcam(dens_model, 'DenseNet-121',    dens_layer)

## Cell 18 — Single Image Prediction

In [ ]:
def predict(image_path, model, model_name):
    img    = Image.open(image_path).convert('RGB')
    tensor = eval_tf(img).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        probs     = torch.softmax(model(tensor), dim=1)[0].cpu().numpy()
        pred_cls  = probs.argmax()

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    fig.suptitle(f'Prediction — {model_name}', fontsize=12, fontweight='bold')
    axes[0].imshow(img); axes[0].set_title('Input X-ray'); axes[0].axis('off')
    colors = ['#F44336' if i == pred_cls else '#2196F3' for i in range(2)]
    bars = axes[1].barh(CLASS_NAMES, probs, color=colors, edgecolor='black')
    axes[1].set_xlim(0, 1.1)
    axes[1].set_title(f'→ {CLASS_NAMES[pred_cls]}  ({probs[pred_cls]*100:.1f}% confidence)')
    for bar, p in zip(bars, probs):
        axes[1].text(p+0.02, bar.get_y()+bar.get_height()/2,
                     f'{p*100:.1f}%', va='center', fontweight='bold')
    plt.tight_layout(); plt.show()
    return CLASS_NAMES[pred_cls], probs

# Test on a random sample
row = df_test.sample(1, random_state=10).iloc[0]
print(f'True label: {CLASS_NAMES[int(row["label"])]}')
predict(row['filepath'], eff_model, 'EfficientNet-B3')

## Cell 19 — Final Summary Table

In [ ]:
print('\n' + '='*60)
print('  FINAL RESULTS — Cardiomegaly Prediction')
print('  Aaditya Sharma et al., MIET')
print('='*60)
print(f'  Dataset  : rahimanshu/cardiomegaly-disease-prediction-using-cnn')
print(f'  Split    : {len(df_train)} train | {len(df_val)} val | {len(df_test)} test')
print(f'  Epochs   : 25  (frozen head → full fine-tune at epoch 10)')
print()
print(f'  {"Metric":<12}  {"EfficientNet-B3":>16}  {"DenseNet-121":>14}')
print('  ' + '-'*46)
for k in ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']:
    e = eff_metrics[k]; d = dens_metrics[k]
    win = '⬅' if e > d else ('  ' if e == d else '        ⬅')
    print(f'  {k:<12}  {e*100:>14.2f}%  {d*100:>12.2f}%  {win}')
print('='*60)
winner = 'EfficientNet-B3' if eff_metrics['Accuracy'] >= dens_metrics['Accuracy'] else 'DenseNet-121'
print(f'  🏆 Best model: {winner}')
print('='*60)

print('\n📁 Output files:')
for f in [
    '/content/efficientnet_b3_best.pth',
    '/content/densenet121_best.pth',
    '/content/dataset_overview.png',
    '/content/sample_images.png',
    '/content/training_curves.png',
    '/content/confusion_matrices.png',
    '/content/roc_curves.png',
    '/content/metrics_comparison.png',
]:
    print(f'  {"✅" if os.path.exists(f) else "❌"}  {f}')

## Cell 20 — Download All Results

In [ ]:
import zipfile
from google.colab import files

zip_path = '/content/cardiomegaly_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in [
        '/content/efficientnet_b3_best.pth',
        '/content/densenet121_best.pth',
        '/content/dataset_overview.png',
        '/content/sample_images.png',
        '/content/training_curves.png',
        '/content/confusion_matrices.png',
        '/content/roc_curves.png',
        '/content/metrics_comparison.png',
    ]:
        if os.path.exists(f):
            z.write(f, arcname=os.path.basename(f))

print('📦 Downloading...')
files.download(zip_path)